# Alpha 指标（py-alpha-lib）全量测试

`py-alpha-lib` 是 Rust 实现的高性能滚动窗口 / 因子算子库（**可选依赖**）：

```bash
py -3.12 -m pip install py-alpha-lib      # 或 pip install "minibt[alpha]"
```

minibt 里的入口：

| 入口 | 说明 |
|---|---|
| `kline.alpha.xxx(...)` | 指标 / 工具函数（返回 `IndSeries`）|
| `kline.ta.alpha_xxx(...)` | CoreFunc 入口（前缀 `alpha_`）|
| `minibt.IndicatorClass.Alpha` | 指标类（78 个方法）|

**数据源约定**：主数据列固定 `FILED.C`（KLine → `close`，IndSeries → 它自己），
方法签名里没有 `input` 参数；需要**第二条序列**的算子才列出其余参数，
**顺序与原指标签名一致**（例如 `ALPHA(input, benchmark, periods)` →
`alpha(benchmark=None, periods=20)`）。

**滚动窗口口径**：alpha 用**扩张窗口**（`min_periods=1`，前几根就有值），
和 pandas 默认 `rolling(n)`（前面是 NaN）不同 —— 对比时只比尾段或加
`min_periods=1` ✓

In [ ]:
from minibt import *
options.set_conversion_mode = "free"


In [ ]:
kline = LocalDatas.test.kline.iloc[:1000]
kline.head()


In [ ]:
import alpha as alpha_lib
print("py-alpha-lib 版本:", getattr(alpha_lib, "__version__", "?"))
methods = sorted(n for n in dir(alpha_lib)
                 if n.isupper() and not n.startswith("FLAG_"))
print("alpha 函数总数:", len(methods))


## 1. 全部函数 × 默认参数

逐个用默认参数调用，记录 `lines` / 最后值 / 失败原因
（需要第二条序列的算子在下一节单独传参）。

In [ ]:
ok_methods, failed_methods = [], []
for name in methods:
    try:
        ind = getattr(kline.alpha, name.lower())()
        print(f"[OK]   {name:<20} lines={list(ind.lines)} 最后={ind.values[-1]}")
        ok_methods.append(name)
    except Exception as error:  # noqa: BLE001
        print(f"[FAIL] {name:<20} {type(error).__name__}: {error}")
        failed_methods.append(name)
print(f"默认参数可运行 {len(ok_methods)} / {len(methods)}")


## 2. 需要额外序列的算子

第二个参数一律传 `kline.open` / `high` / `low` 的 `np.ndarray`。

In [ ]:
ref = kline.open.values          # 参考序列（np.ndarray）✓
extra = {
    "cross": dict(b=ref),
    "rcross": dict(b=ref),
    "longcross": dict(b=ref, n=5),
    "rlongcross": dict(b=ref, n=5),
    "max": dict(b=ref),
    "min": dict(b=ref),
    "corr2": dict(y=ref, periods=20),
    "cov": dict(y=ref, periods=20),
    "beta": dict(benchmark=ref, periods=20),
    "alpha": dict(benchmark=ref, periods=20),
    "regresi": dict(x=ref, periods=20),
    "regbeta": dict(x=ref, periods=20),
    "where": dict(a=kline.high.values, b=kline.low.values),
    "scan_add": dict(condition=kline.close.values > kline.open.values),
    "scan_mul": dict(condition=kline.close.values > kline.open.values),
    "sumif": dict(condition=kline.close.values > kline.open.values, periods=20),
    "filter": dict(periods=20),
    "group_rank": dict(input=kline.close.values),
    "group_zscore": dict(input=kline.close.values),
    "neutralize": dict(input=kline.close.values),
    "fret": dict(close=kline.close.values, is_calc=kline.close.values),
    "bw_split": dict(dividend=kline.close.values * 0,
                     transfer_shares=kline.close.values * 0,
                     right_shares=kline.close.values * 0,
                     right_price=kline.close.values),
    "bw_split_factor": dict(dividend=kline.close.values * 0,
                            transfer_shares=kline.close.values * 0,
                            right_shares=kline.close.values * 0,
                            right_price=kline.close.values),
    "fw_split": dict(dividend=kline.close.values * 0,
                     transfer_shares=kline.close.values * 0,
                     right_shares=kline.close.values * 0,
                     right_price=kline.close.values),
    "fw_split_factor": dict(dividend=kline.close.values * 0,
                            transfer_shares=kline.close.values * 0,
                            right_shares=kline.close.values * 0,
                            right_price=kline.close.values),
}
extra_failed = []
for name, kwargs in extra.items():
    try:
        ind = getattr(kline.alpha, name)(**kwargs)
        print(f"[OK]   {name:<18} lines={list(ind.lines)} 最后={ind.values[-1]}")
    except Exception as error:  # noqa: BLE001
        print(f"[FAIL] {name:<18} {type(error).__name__}: {error}")
        extra_failed.append(name)


## 3. 可画线指标（desk 选择器里的 40 个）

`NON_KLINE_INDICATORS` 里的算子上不了图，其余都能直接挂到 K 线上。

In [ ]:
import minibt
library = minibt.IndicatorClass.Alpha
skip = set(getattr(library, "NON_KLINE_INDICATORS", []))
draw = [name for name in methods if name.lower() not in skip]
print(f"可画线 {len(draw)} / 算子 {len(skip)}")
for name in draw:
    ind = getattr(kline.alpha, name.lower())()
    print(f"{name:<20} overlap={ind.overlap} lines={list(ind.lines)} 最后={ind.values[-1]}")


## 4. CoreFunc 入口（前缀 `alpha_`）

`kline.alpha.ma(periods=10)` 与 `kline.ta.alpha_ma(periods=10)` 等价。

In [ ]:
ma1 = kline.alpha.ma(periods=10)
ma2 = kline.ta.alpha_ma(periods=10)
print("两者一致:", bool((ma1.values == ma2.values).all()))

# IndSeries 上也能直接用（数据源 = 它自己）
line = kline.close
print("IndSeries 上 alpha.ma:", line.alpha.ma(periods=5).values[-3:])


## 5. 参数顺序（和原指标签名一致）

原：`ALPHA(input, benchmark, periods)` → minibt：`(benchmark=None, periods=20)`；
原：`BW_SPLIT(price, dividend, transfer_shares, right_shares, right_price)`
→ minibt：`(dividend, transfer_shares, right_shares, right_price)`。

In [ ]:
import inspect
from minibt.core import CoreFunc
for name in ("alpha_alpha", "alpha_beta", "alpha_cross", "alpha_bw_split",
             "alpha_corr2", "alpha_regresi", "alpha_fret"):
    print(name, inspect.signature(getattr(CoreFunc, name)))


## 6. 与 pandas 对齐校验

`MA / SUM / HHV / LLV` 用 `min_periods=1`（扩张窗口）；
`STDDEV / VAR / ZSCORE` 用 `ddof=1` + `min_periods=20` ✓

In [ ]:
import numpy as np
import pandas as pd

close = pd.Series(kline.close.values, dtype=float)
tail = -50          # 只比尾段（避免扩张窗口的口径差异）


def compare(label, got, expect):
    left = np.asarray(got, dtype=float)[tail:]
    right = np.asarray(expect, dtype=float)[tail:]
    same = np.allclose(left, right, equal_nan=True, rtol=1e-6, atol=1e-6)
    print(f"{label:<12} 与 pandas 一致: {same}")


compare("MA20", kline.alpha.ma(periods=20).values,
        close.rolling(20, min_periods=1).mean())
compare("SUM5", kline.alpha.sum(periods=5).values,
        close.rolling(5, min_periods=1).sum())
compare("HHV10", kline.alpha.hhv(periods=10).values,
        close.rolling(10, min_periods=1).max())
compare("LLV10", kline.alpha.llv(periods=10).values,
        close.rolling(10, min_periods=1).min())
compare("STDDEV20", kline.alpha.stddev(periods=20).values,
        close.rolling(20, min_periods=20).std(ddof=1))
compare("VAR20", kline.alpha.var(periods=20).values,
        close.rolling(20, min_periods=20).var(ddof=1))
mean20 = close.rolling(20, min_periods=20).mean()
std20 = close.rolling(20, min_periods=20).std(ddof=1)
compare("ZSCORE20", kline.alpha.zscore(periods=20).values,
        ((close - mean20) / std20))


## 7. 默认参数失败的清单（不是 bug，是缺第二条序列）

In [ ]:
failed_methods


## 8. 传参后仍失败的（如果有）

大多数是拆股 / 分组类需要业务语义（`category` 用整数分组码等）。

In [ ]:
extra_failed


## 小结

- `kline.alpha.*` 是 py-alpha-lib（Rust）的门面：小写方法名 + 原库参数顺序（去掉第一个输入序列）；
- 需要第二条序列的算子（`cross` / `corr2` / `beta` …）传 `b=` / `y=`；
- `NON_KLINE_INDICATORS` 里的算子上不了图，可用于选股 / 因子打分；
- CoreFunc 入口等价：`kline.alpha.ma(periods=10)` == `kline.ta.alpha_ma(periods=10)`。

## 下一步

- 因子全量：`tutorials/test/alpha101.ipynb` / `tutorials/test/alpha191.ipynb`（292 个）；
- 因子建模：`tutorials/test/factors.ipynb`；
- 文档：[`minibt_api_reference/factors.md`](../minibt_docs/docs/minibt_api_reference/factors.md)。
